In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import torch
import yaml

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import load_data
from episode_sampler import EpisodeSampler
from encoder import Encoder
from mamlnet import MAMLNet
from utils import deterministic
from maml_utils import train_maml, plot_maml_history

In [ ]:
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## MAML

In [ ]:
data = "Mnist"
experiment_parameters = config["maml"]
n_way = experiment_parameters["n_way"]
k_shot = experiment_parameters["k_shot"]
q_query = experiment_parameters["q_query"]
meta_batch_size = experiment_parameters["meta_batch_size"]
inner_updates = experiment_parameters["inner_updates"]
meta_iterations = experiment_parameters["meta_iterations"]
in_channels = experiment_parameters["in_channels"]
hidden_dim = experiment_parameters["hidden_dim"]
embedding_dim = experiment_parameters["embedding_dim"]
alpha = experiment_parameters["alpha"]
beta = experiment_parameters["beta"]


train_images, train_labels = load_data("train", config[data]["dataset_output_dir"])
val_images, val_labels = load_data("val", config[data]["dataset_output_dir"])

encoder = Encoder(in_channels=in_channels, hidden_dim=hidden_dim, embedding_dim=embedding_dim)
encoder.to(device)
model = MAMLNet(encoder, n_way=n_way, protomaml=False)
model.to(device)

train_sampler = EpisodeSampler(images=train_images, labels=train_labels,
                               n_way=n_way, k_shot=k_shot, q_query=q_query)

val_sampler = EpisodeSampler(images=val_images, labels=val_labels,
                             n_way=n_way, k_shot=k_shot, q_query=q_query)

optimizer = torch.optim.Adam(model.parameters(), lr=beta)

In [ ]:
deterministic(config["xtra"]["seed"])
maml_history = train_maml(model, train_sampler, val_sampler, optimizer, meta_iterations, meta_batch_size, inner_updates, alpha, device)

In [ ]:
path = "../models/weights/maml.pth"
model.save(path)

In [ ]:
path = "../results/maml_history.yaml"
with open(path, "w") as f:
    yaml.dump(maml_history, f)

## ProtoMAML

In [ ]:
data = "Mnist"
experiment_parameters = config["protomaml"]
n_way = experiment_parameters["n_way"]
k_shot = experiment_parameters["k_shot"]
q_query = experiment_parameters["q_query"]
meta_batch_size = experiment_parameters["meta_batch_size"]
inner_updates = experiment_parameters["inner_updates"]
meta_iterations = experiment_parameters["meta_iterations"]
in_channels = experiment_parameters["in_channels"]
hidden_dim = experiment_parameters["hidden_dim"]
embedding_dim = experiment_parameters["embedding_dim"]
alpha = experiment_parameters["alpha"]
beta = experiment_parameters["beta"]


train_images, train_labels = load_data("train", config[data]["dataset_output_dir"])
val_images, val_labels = load_data("val", config[data]["dataset_output_dir"])

encoder = Encoder(in_channels=in_channels, hidden_dim=hidden_dim, embedding_dim=embedding_dim)
encoder.to(device)
model = MAMLNet(encoder, n_way=n_way, protomaml=True)
model.to(device)

train_sampler = EpisodeSampler(images=train_images, labels=train_labels,
                               n_way=n_way, k_shot=k_shot, q_query=q_query)

val_sampler = EpisodeSampler(images=val_images, labels=val_labels,
                             n_way=n_way, k_shot=k_shot, q_query=q_query)

optimizer = torch.optim.Adam(model.parameters(), lr=beta)

In [ ]:
deterministic(config["xtra"]["seed"])
protomaml_history = train_maml(model, train_sampler, val_sampler, optimizer, meta_iterations, meta_batch_size, inner_updates, alpha, device)

In [ ]:
path = "../models/weights/protomaml.pth"
model.save(path)

In [ ]:
path = "../results/protomaml_history.yaml"
with open(path, "w") as f:
    yaml.dump(protomaml_history, f)

## Plots

### MAML

In [ ]:
path = "../results/maml_history.yaml"
with open(path, "r") as f:
    maml_history = yaml.safe_load(f)

In [ ]:
plot_maml_history(maml_history, "../images/maml/maml_training.png")

### ProtoMAML

In [ ]:
path = "../results/protomaml_history.yaml"
with open(path, "r") as f:
    protomaml_history = yaml.safe_load(f)

In [ ]:
plot_maml_history(protomaml_history, "../images/protomaml/protomaml_training.png")